In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display
import getpass
import os
import re
import time

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        print(log.read_text())
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


def outputs_ready(paths, submitted_at):
    if all(path.is_file() and path.stat().st_mtime >= submitted_at for path in paths):
        return True
    print("New results are not available yet. Wait for this job to finish, then rerun this cell.")
    return False

print(f"Repository: {repo_root}")

# GPU Execution Model


> **Overview**
>
> 
>  * **Time:** 60 min
> 
>  1. Learn the CUDA execution model.
>  1. Understand the difference between CUDA kernels and functions.
>  1. Learn how to launch CUDA kernels and manage thread indexing.

CUDA (Compute Unified Device Architecture) is a parallel computing platform and programming model developed by NVIDIA. It allows developers to use a
CUDA-enabled GPU for general-purpose processing, leveraging its massive parallelism capabilities.

## CUDA Kernels vs Functions


In CUDA, both kernels and functions are used to define code behavior, but they have distinct purposes, execution models, and qualifiers.
CUDA supports three types of functions based on qualifiers:

**CUDA Function Types**

| Qualifier | Meaning |
| --- | --- |
| `__global__` | Defines a *kernel* function that runs on the device (GPU) and is called from the host (CPU). |
| `__device__` | Defines a function that runs on the device and can only be called from other device or global functions. |
| `__host__` | Defines a function that runs on the host (CPU). This is the default for C/C++ functions. |
| `__host__ __device__` | The function can be compiled for and called from both host and device. Often used for small inline functions or utilities. |

### CUDA Kernel


A kernel is a special type of function that is executed in parallel by many GPU threads. Kernels are defined using the `__global__` keyword and are launched
using triple angle brackets syntax.

```cpp
__global__ void add(int *a, int *b, int *c) {
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}

// Launch kernel with 1 block of 256 threads
add<<<1, 256>>>(d_a, d_b, d_c);
```

### CUDA Device Function


Device functions run on the GPU but are called only from other device or global functions. They are useful for modularizing GPU code.

```cpp
__device__ int add(int x, int y) {
    return x + y;
}
```

### Host Function


Host functions are standard C/C++ functions that execute on the CPU. They typically manage memory and kernel launches.

```cpp
void initArrays(int *a, int *b) {
    for (int i = 0; i < N; ++i) {
        a[i] = i;
        b[i] = i * 2;
    }
}
```

**Difference Between CUDA Kernel and Function**

| Feature | CUDA Kernel (`__global__`) | CUDA Function (`__device__`, `__host__`) |
| --- | --- | --- |
| Purpose | Entry point for GPU parallel execution | General subroutine for computation |
| Runs On | GPU | CPU or GPU |
| Called From | Host (CPU) | Device or Host |
| Invocation | `<<<grid, block>>>` | Standard function call |
| Return Type | Must be `void` | Can return values |
| Parallelism | Runs across many GPU threads | Runs per thread (device) or serially (host) |
| Use Case | Massively parallel tasks | Helper or utility routines |

## Execution Model


In CUDA, parallelism is achieved through a hierarchy of **grids**, **blocks**, and **threads**. When launching a kernel, you specify the grid and block
configuration using the syntax:

```cpp
kernel_name<<<numBlocks, threadsPerBlock>>>(args);
```

The execution model of CUDA is based on a hierarchy of threads organized into blocks and grids. This model allows for massive parallelism by executing many
threads concurrently on the GPU. The basic structure is as follows:

**CUDA Execution Model**

| Level | Description | Example |
| --- | --- | --- |
| Grid | A grid is a collection of blocks that execute a kernel. | A grid can be 1D, 2D, or 3D. |
| Block | A block is a group of threads that execute together and can share memory. | Blocks can also be 1D, 2D, or 3D. |
| Thread | The smallest unit of execution. Each thread executes the same kernel code but operates on different data. | Threads are identified by their unique thread index within a block. |

Example (1D Launch):

```cpp
add<<<4, 256>>>(a, b, c);
```

In the example above, the kernel `add()` is launched with 1-D grid which has 4 blocks, each block containing 256 threads. This launches a kernel with:

* 4 blocks in the grid
* 256 threads in each block
* Total threads = 4 × 256 = **1024 threads**

Each performing the `add()` operation.

The same kernel can be launched with different configurations, such as 2D or 3D grids and blocks, to suit the problem's dimensionality. For example,
a 2D grid with 2D blocks might look like this:

```cpp
kernel_name<<<dim3(2, 2), dim3(16, 16)>>>(args);
```

This launches a kernel with:
* 2 × 2 = 4 blocks in the grid (2D)
* Each block has 16 × 16 = 256 threads
* Total threads = 4 × 256 = **1024 threads**

Similarly we can launch a 3D kernel:

```cpp
kernel_name<<<dim3(2, 2, 2), dim3(4, 4, 4)>>>(args);
```

This launches a kernel with:

* 2 × 2 × 2 = 8 blocks in the grid (3D)
* Each block has 4 × 4 × 4 = 64 threads
* Total threads = 8 × 64 = **512 threads**


> **Explanation**
>
> 
> `dim3` is a built-in C++ struct used to define the dimensions of a grid or a block when launching a kernel.
> It's essentially a convenience structure that stores 3D dimensions (x, y, z), where:
> 
>  * x is required
>  * y and z default to 1 if not specified

## Thread and Block Indexing


Each thread is aware of its position in the block and the grid using built-in variables:

**Built-in CUDA Thread Identifiers**

| Variable | Meaning |
| --- | --- |
| `threadIdx.x`, `threadIdx.y`, `threadIdx.z` | Thread’s index within the block |
| `blockIdx.x`, `blockIdx.y`, `blockIdx.z` | Block’s index within the grid |
| `blockDim.x`, `blockDim.y`, `blockDim.z` | Number of threads per block in each dimension |
| `gridDim.x`, `gridDim.y`, `gridDim.z` | Number of blocks in the grid in each dimension |

### Computing Global Thread Index


In most cases when you are working with CUDA, you will need to compute a unique global thread index that identifies each thread across the entire grid.
This is essential for accessing global memory or performing operations that require a unique identifier for each thread.

In [ ]:
# CUDA Thread Indexing
display(Image(filename=str(repo_root / "docs/source/tutorial/figs/thread-index.drawio.png")))

**CUDA Thread Indexing Example**

| Dimension | Example Code |
| --- | --- |
| 1D | `int idx = blockIdx.x * blockDim.x + threadIdx.x;` |
| 2D | `int idx = (blockIdx.x * blockDim.x + threadIdx.x) + (blockIdx.y * blockDim.y + threadIdx.y) * gridDim.x * blockDim.x;` |
| 3D | `int idx = (blockIdx.x * blockDim.x + threadIdx.x) + (blockIdx.y * blockDim.y + threadIdx.y) * gridDim.x * blockDim.x + (blockIdx.z * blockDim.z + threadIdx.z) * gridDim.x * gridDim.y * blockDim.x * blockDim.y;` |

This indexing allows each thread to access its unique portion of data in global memory, enabling efficient parallel processing.

## Why Global Thread Index Matters


In CUDA, threads run the same code (SIMT model) - which stands for Single Instruction, Multiple Threads. Threads are organized into warps within a block. Warps execute instructions for their active threads; different warps can progress independently and operate on different data.

So without a way to uniquely identify itself, every thread would operate on the same memory location. The  **global thread index** helps distribute work across
threads so they operate on **independent data**.

Suppose we want to add two arrays `a` and `b` of size `n`, and store the result in `c`. The CUDA kernel would look like this:

```cpp
__global__ void add(int *a, int *b, int *c, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x; // Compute global thread index
    if (idx < n) { // Ensure we don't go out of bounds
        c[idx] = a[idx] + b[idx]; // Perform addition
    }
}
```

Kernel Launch for the Example would look like this:

```cpp
int n = 1000;
int threadsPerBlock = 256;
int numBlocks = (n + threadsPerBlock - 1) / threadsPerBlock;

add_arrays<<<numBlocks, threadsPerBlock>>>(a, b, c, n);
```

## Exercise 03: GPU Execution Model

**Time: 30 min**

Follow the four steps in this order, using two blocks of four threads.
[Exercise instructions](../docs/source/tutorial/exercises/03-gpu-exec-model.rst).

Run the setup cell first. Build the exercises once in
[02-gpu.ipynb](02-gpu.ipynb) before submitting jobs from this notebook.
After editing CUDA source, return to that notebook to rebuild.

Submission cells use the same `qsub` commands as the exercise documentation,
from `job-script`, and immediately show `qstat`. A submitted job may still be queued.
Use the monitoring cell to check again; rerunning a submission cell creates another job.
Read the PBS output and inspect generated files after the job finishes.

### 1. Print the thread index

Every thread prints its local index, block index, block dimension and grid dimension. Expect eight lines in any order: local indices 0–3 repeat in each block; block dimension is 4 and grid dimension is 2. Why does each block have a thread 0?

<a href="../exercises/03-gpu-exec-model/01-print-thread-index/print-thread-index.cu" target="_blank" rel="noopener noreferrer">Open print-thread-index.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
step1_submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 03-01-print-thread-index.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
step1_job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if step1_job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


In [ ]:
!qstat -u "{username}"
show_job_output("03-01-print-thread-index", step1_job_id)


### 2. Double using the thread index

This intentionally incorrect program uses only `threadIdx.x` to double an eight-element array. Both blocks address elements 0–3; neither addresses elements 4–7. The overlapping updates also create a data race: do not assume the first four values are reliably doubled twice. Why does the bounds check fail to fix the indexing bug?

<a href="../exercises/03-gpu-exec-model/02-double-thread-index/double-thread-index.cu" target="_blank" rel="noopener noreferrer">Open double-thread-index.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
step2_submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 03-02-double-thread-index.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
step2_job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if step2_job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


In [ ]:
!qstat -u "{username}"
show_job_output("03-02-double-thread-index", step2_job_id)


### 3. Print the global thread index

Inspect `blockIdx.x * blockDim.x + threadIdx.x`. Global indices run from 0 to 7 even though local indices repeat. Why does thread 0 in block 1 have global index 4?

<a href="../exercises/03-gpu-exec-model/03-print-global-index/print-global-index.cu" target="_blank" rel="noopener noreferrer">Open print-global-index.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
step3_submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 03-03-print-global-index.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
step3_job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if step3_job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


In [ ]:
!qstat -u "{username}"
show_job_output("03-03-print-global-index", step3_job_id)


### 4. Double using the global index

Now each array element has one writer. Expect `2 4 6 8 10 12 14 16` and `PASS`. Compare with step 2. What happens with one block of eight threads?

<a href="../exercises/03-gpu-exec-model/04-double-global-index/double-global-index.cu" target="_blank" rel="noopener noreferrer">Open double-global-index.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
step4_submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 03-04-double-global-index.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
step4_job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if step4_job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


In [ ]:
!qstat -u "{username}"
show_job_output("03-04-double-global-index", step4_job_id)


### 5. Explore thread indexing inside the notebook

Edit `grid` and `block`, then rerun this cell. The SVG is calculated from formulas
by the exercise's `draw-indices.py` helper and rendered inline. It needs no GPU
or job submission. It is not a measurement of GPU execution.

<a href="../exercises/03-gpu-exec-model/05-thread-indexing/draw-indices.py" target="_blank" rel="noopener noreferrer">Open draw-indices.py</a>

In [ ]:
import runpy

grid = (2, 2)   # Number of blocks in x and y.
block = (2, 2)  # Threads per block in x and y.
assert len(grid) == len(block) == 2
assert all(type(d) is int and 1 <= d <= 16 for d in (*grid, *block))

tools = runpy.run_path(str(repo_root / "exercises/03-gpu-exec-model/05-thread-indexing/draw-indices.py"))
thread_data = list(tools["reference_records"](*grid, *block))
output_dir = repo_root / "notebook-output/03-gpu-exec-model"
output_dir.mkdir(parents=True, exist_ok=True)
image_path = tools["draw"](*grid, *block, output_dir / "custom.svg", records=thread_data)
display(SVG(filename=str(image_path)))
thread_data[:4]


### Check the queue again

Use this cell to monitor the four step jobs without resubmitting them. Rerun each
job-output cell above after its job finishes. The inline explorer uses formulas
and does not submit GPU work; changing its variables does not change the CUDA sources.

In [ ]:
!qstat -u "{username}"
